In [0]:
%pip install shapely pyproj

In [0]:
from pyspark.sql import functions as F

# read the raw csv file
df_raw_csv = spark.read.csv("/Volumes/workspace/default/solar_raw/de_hh_up_dachseiten_EPSG_25832.csv", sep=';', header=True)

# read the geojson file (single FeatureCollection object -> multiLine mode)
df_raw_geojson = spark.read.option("multiLine", "true").json("/Volumes/workspace/default/solar_raw/hamburg_city_districts.geojson")

# explode the features array into one row per district
df_raw_district_data = df_raw_geojson.select(F.explode("features").alias("feature")) \
    .select("feature.properties.Stadtteil", "feature.geometry.type", "feature.geometry.coordinates")

In [0]:
# Save the geojson dataframe as a delta table
df_raw_district_data.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("hamburg_city_districts_raw")

In [0]:
df_raw_csv.printSchema() # print the schema of the DataFrame


In [0]:
df_raw_csv.show() # print the first 20 rows of the DataFrame

In [0]:
# change the columns type according to their stored values
from pyspark.sql import functions as F

columns = ["area", "aspect", "aufstd", "buildingid", "ertkwp_k", "ertkwp_ka", "ertkwha_k", "ertkwha_ka", "percentms", "percentmsa", "power", "pvarea", "pvareat", "roofid", "schatten","schattena", "slope"]
for col in columns:
  df_raw_csv = df_raw_csv.withColumn(col, F.expr(f"try_cast({col} as double)"))
df_raw_csv.printSchema()

In [0]:
# check the id column
df_raw_csv.select("_fid").toPandas()

In [0]:
from pyspark.sql import functions as F

# replace the id column with a new one and keep only numbers
df_raw_csv = df_raw_csv.withColumn("_fid", F.expr("try_cast(regexp_replace(_fid, '[^0-9]', '') as int)"))
df_raw_csv = df_raw_csv.withColumnRenamed("_fid", "id")
df_raw_csv.select("id").toPandas()

In [0]:
# Save the dataframe as a delta table
df_raw_csv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("solar_raw")

# **Silver Table : Data Cleaning Pipeline**

Drop columns which is not important for the model and prediction.
1. einigung_t : duplicate column
2. percentms : rediation percentage
3. ertkwha_K : unusefull information

In [0]:
# create meaningfull column for the eignung : Suitability class
df_raw_csv = df_raw_csv.withColumnRenamed("eignung", "suitability_class")

# map column values with numbers
df_raw_csv = df_raw_csv.withColumn("suitability_class", F.expr("case when suitability_class = 'Eignung 8 (kein Gebäude erkannt (Boden))' then 8 when suitability_class = 'Eignung 6 (geringe Einstrahlung / weniger geeignet / vor Ort zu prüfen)' then 6 when suitability_class = 'Eignung 5 (geringe Einstrahlung / weniger geeignet)' then 5 when suitability_class = 'Eignung 4 (mäßige Einstrahlung / mittelgeeignet)' then 4 when suitability_class = 'Eignung 3 (geeignet, mittlere Einstrahlung)' then 3 when suitability_class = 'Eignung 2 (geeignet, hohe Einstrahlung)' then 2 when suitability_class = 'Eignung 1 (geeignet, sehr hohe Einstrahlung)' then 1 else 0 end"))


In [0]:
# drop columns which are not important for the model and prediction
df_raw_csv = df_raw_csv.drop("eignung_t", "percentms", "ertkwha_k")

In [0]:
# clean the data for the silver layer
df_silver = (
    df_raw_csv
    # keep only usable roof records: drop class 0 (insufficient data quality)
    # and class 8 (no building recognized / ground surface)
    .filter("suitability_class in (1, 2, 3, 4, 5, 6)")
    # remove impossible area outliers (> 100,000 m2; real roofs max ~68,000 m2)
    .filter("area <= 100000")
    # drop the constant column (always 'EPSG:25832')
    .drop("coordinate_reference_system")
    # safety net: remove duplicates if any
    .dropDuplicates()
)

In [0]:
# verify the cleaning result
print("rows before cleaning:", df_raw_csv.count())
print("rows after cleaning:", df_silver.count())
display(df_silver.groupBy("suitability_class").count().orderBy("suitability_class"))

In [0]:
# save the cleaned dataframe as the silver delta table
df_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("solar_silver")

In [0]:
# spatial join: assign each roof to its Hamburg district (Stadtteil)
import json
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from shapely.strtree import STRtree
from pyproj import Transformer

# districts are in WGS84, roofs in EPSG:25832 -> transform the 104 district polygons
districts_to_utm = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True).transform

with open("/Volumes/workspace/default/solar_raw/hamburg_city_districts.geojson", "r", encoding="utf-8") as f:
    gj = json.load(f)

district_geoms = [shp_transform(districts_to_utm, shape(feat["geometry"])) for feat in gj["features"]]
district_names = [feat["properties"]["Stadtteil"] for feat in gj["features"]]

# roof polygons -> centroids, then point-in-polygon join against the district STRtree
roof_pd = df_silver.select("id", "geom").toPandas()
roof_centroids = shapely.centroid(shapely.from_wkt(roof_pd["geom"].values))

partition_tree = STRtree(district_geoms)
within_idx = partition_tree.query(roof_centroids, predicate="within")

# roofs whose centroid falls on a district border: assign the nearest district (within 500 m)
matched = np.zeros(len(roof_pd), dtype=bool)
matched[within_idx[0]] = True
unmatched_pos = np.where(~matched)[0]
nearest_idx = partition_tree.query_nearest(roof_centroids[unmatched_pos], max_distance=500)

map_roof = np.concatenate([within_idx[0], unmatched_pos[nearest_idx[0]]])
map_dist = np.concatenate([within_idx[1], nearest_idx[1]])
mapping_pd = pd.DataFrame({
    "id": roof_pd["id"].values[map_roof],
    "stadtteil": np.array(district_names)[map_dist],
}).drop_duplicates(subset="id", keep="first")

df_roof_district = spark.createDataFrame(mapping_pd)
df_gold = df_silver.join(df_roof_district, on="id", how="left")
display(df_gold.select("id", "stadtteil", "suitability_class", "area", "power").limit(5))

In [0]:
# verify: every roof has exactly one district
print("roofs:", df_gold.count())
print("roofs without district:", df_gold.filter(F.col("stadtteil").isNull()).count())
display(df_gold.groupBy("stadtteil").count().orderBy(F.desc("count")).limit(10))

In [0]:
# save the spatially enriched table
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("solar_roofs_with_district")

In [0]:
%sql
-- aggregate the power potential per district
SELECT
  stadtteil,
  COUNT(*) AS roofs,
  ROUND(SUM(power) / 1000, 1) AS total_power_mwp,
  ROUND(SUM(CASE WHEN suitability_class IN (1, 2, 3) THEN power END) / 1000, 1) AS suitable_power_mwp,
  SUM(CASE WHEN suitability_class IN (1, 2, 3) THEN 1 ELSE 0 END) AS suitable_roofs,
  ROUND(SUM(pvarea), 0) AS total_pv_area_m2
FROM workspace.default.solar_roofs_with_district
GROUP BY stadtteil
ORDER BY total_power_mwp DESC

In [0]:
%sql
-- top 10 districts: total vs suitable power, reshaped for grouped bars
WITH district_totals AS (
  SELECT
    stadtteil,
    ROUND(SUM(power) / 1000, 1) AS total_power_mwp,
    ROUND(SUM(CASE WHEN suitability_class IN (1, 2, 3) THEN power END) / 1000, 1) AS suitable_power_mwp
  FROM workspace.default.solar_roofs_with_district
  GROUP BY stadtteil
  ORDER BY total_power_mwp DESC
  LIMIT 10
), reshaped AS (
  SELECT stadtteil, total_power_mwp AS sort_key, 'Total power' AS measure, total_power_mwp AS power_mwp FROM district_totals
  UNION ALL
  SELECT stadtteil, total_power_mwp AS sort_key, 'Suitable power (class 1-3)' AS measure, suitable_power_mwp AS power_mwp FROM district_totals
)
SELECT stadtteil, measure, power_mwp
FROM reshaped
ORDER BY sort_key DESC, measure DESC

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- gold table: power potential aggregated per district
CREATE OR REPLACE TABLE workspace.default.solar_gold AS
SELECT
  stadtteil,
  COUNT(*) AS roofs,
  ROUND(SUM(power) / 1000, 1) AS total_power_mwp,
  ROUND(SUM(CASE WHEN suitability_class IN (1, 2, 3) THEN power END) / 1000, 1) AS suitable_power_mwp,
  SUM(CASE WHEN suitability_class IN (1, 2, 3) THEN 1 ELSE 0 END) AS suitable_roofs,
  ROUND(SUM(pvarea), 0) AS total_pv_area_m2
FROM workspace.default.solar_roofs_with_district
GROUP BY stadtteil

In [0]:
# train a suitability prediction model
# target: suitability_class (1/2/3 = suitable tiers, 6 = marginal)
# features: roof geometry, shadowing and radiation inputs + district.
# PV output columns (power, pvarea, pvareat, ertkwp_k, ertkwp_ka, ertkwha_ka) are EXCLUDED:
# they are computed consequences of the same assessment that produced the label (leakage).
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

FEATURES_NUM = ["area", "aspect", "aufstd", "slope", "schatten", "schattena", "percentmsa"]
FEATURES_CAT = ["stadtteil"]
TARGET = "suitability_class"

def make_pipeline():
    preprocess = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), FEATURES_CAT)],
        remainder="passthrough",
    )
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.1,
            categorical_features=[0],  # stadtteil is column 0 after preprocessing
            random_state=42,
        )),
    ])

model_data = spark.table("workspace.default.solar_roofs_with_district").select(FEATURES_NUM + FEATURES_CAT + [TARGET]).toPandas()
X = model_data[FEATURES_NUM + FEATURES_CAT]
y = model_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("train rows:", len(X_train), "| test rows:", len(X_test))

model = make_pipeline()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1 macro:", round(f1_score(y_test, y_pred, average="macro"), 4))
print(classification_report(y_test, y_pred))
cm_fig = ConfusionMatrixDisplay.from_predictions(y_test, y_pred).figure_

In [0]:
# log the model to MLflow (refit on all rows for the final artifact)
import mlflow
from sklearn.base import clone
from mlflow.models import infer_signature

final_model = make_pipeline()
final_model.fit(X, y)

signature = infer_signature(X.head(100), final_model.predict(X.head(100)))

mlflow.set_experiment("/Users/dhrupti.pambhar1999@gmail.com/solarscope_hamburg/solar_suitability_model")
with mlflow.start_run(run_name="hist_gradient_boosting_v1"):
    mlflow.log_params({"model": "HistGradientBoostingClassifier", "max_iter": 300, "learning_rate": 0.1,
                       "features": len(FEATURES_NUM) + len(FEATURES_CAT), "excluded_leakage": "power, pvarea, pvareat, ertkwp_k, ertkwp_ka, ertkwha_ka"})
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
        "n_train": len(X_train), "n_test": len(X_test), "n_final_fit": len(X),
    })
    mlflow.log_figure(cm_fig, "confusion_matrix.png")
    model_info = mlflow.sklearn.log_model(final_model, name="model", signature=signature, input_example=X.head(3))

print("logged model:", model_info.model_uri)

In [0]:
# register the validated model in Unity Catalog
mlflow.set_registry_uri("databricks-uc")
registered_version = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="workspace.default.solar_suitability_model",
    await_registration_for=300,
)
print("registered:", registered_version.name, "version", registered_version.version)

In [0]:
# Model v2 - rebuilt to match the project guide
# why?: the guide says the model must NOT use radiation columns (percentms, percentmsa, schattena).
#        they leak the label: the suitability class is basically computed FROM radiation, so a model
#        using them is like revising for a test with the answer sheet on the desk.
# use?: only features a roof owner knows beforehand: size, direction, tilt, shading, roof type, district.
# need?: the train/test split is BY DISTRICT (every 5th district held out) - roofs in the same
#        district are similar, so a random split would let the model "recognise" neighbourhoods.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

FEATURES_NUM = ["area", "aspect", "aufstd", "slope", "schatten"]
FEATURES_CAT = ["stadtteil"]
FEATURES = FEATURES_NUM + FEATURES_CAT
TARGET = "suitability_class"

data = spark.table("workspace.default.solar_roofs_with_district").select(FEATURES + [TARGET]).toPandas()

# district-based split: every 5th district (alphabetically) becomes the test set
districts = sorted(data["stadtteil"].unique())
test_districts = districts[::5]
train_df = data[~data["stadtteil"].isin(test_districts)]
test_df = data[data["stadtteil"].isin(test_districts)]
print(f"train: {len(train_df)} rows | test: {len(test_df)} rows from {len(test_districts)} held-out districts")


def make_pipeline():
    # need: one shared recipe so training here and the app build the model the same way
    return Pipeline([
        ("preprocess", ColumnTransformer(
            # turn district names into numbers - trees can only work with numbers
            [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), FEATURES_CAT)],
            remainder="passthrough")),
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)),
    ])


model = make_pipeline()
model.fit(train_df[FEATURES], train_df[TARGET])
y_pred = model.predict(test_df[FEATURES])
print("accuracy:", accuracy_score(test_df[TARGET], y_pred))
print("F1 macro:", f1_score(test_df[TARGET], y_pred, average="macro"))
print(classification_report(test_df[TARGET], y_pred))

# what did the model learn? (check it makes physical sense)
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=["stadtteil"] + FEATURES_NUM).sort_values(ascending=False)
print(importances.round(3))